In [43]:
import time
import numpy as np
import scipy.io
import scipy.sparse as sp
import scipy.sparse.linalg as spla

from spectral import *
from logdet import *
from leja_action import *
from divided_differences import *
from hutchpp import *

##for pentadiagonale matrices, uncomment the following


# # Create a pentadiagonal matrix of size 20x20
# np.random.seed(42)
# size =50000
# diagonals = [np.random.rand(size), np.random.rand(size-1), np.random.rand(size-2),
#              np.random.rand(size-1), np.random.rand(size-2)]
# Q = sp.diags(diagonals, [0, 1, 2, -1, -2], format='csr')

# # Ensure the matrix is positive semi-definite
# Q = Q + Q.T
# Q = Q + size * sp.eye(size)


def load_uf_matrix_mat(file_path: str):
    data = scipy.io.loadmat(file_path)
    problem = data["Problem"]
    A = problem["A"][0, 0]
    # UF .mat often stores sparse as scipy sparse already; enforce CSR
    if sp.issparse(A):
        return A.tocsr()
    return sp.csr_matrix(A)

if __name__ == "__main__":
    Q = load_uf_matrix_mat("data/ecology2.mat")
    n = Q.shape[0]
    print("Loaded Q:", Q.shape, "nnz =", Q.nnz)
    t0 = time.perf_counter()
    m_leja = 500
    m_hutchpp = 12
    tol = 1e-6

    Leja_X_all = np.loadtxt("data/Leja_10000.txt")
    Leja_X = Leja_X_all[: m_leja + 1]

    spectral = FocalIntervalEstimator(min_floor=1e-12)
    dd = DividedDifferencesLog(taylor_degree=250)
    leja = LejaLogAction(divided_diff=dd)
    hpp = HutchPP(rng=42)

    estimator = LogDetEstimatorHutchPPLeja(spectral=spectral, leja_action=leja, hutchpp=hpp)

    
    est, info = estimator.estimate(Q, leja_points=Leja_X, m_leja=m_leja, m_hutchpp=m_hutchpp, tol=tol)
    t1 = time.perf_counter()

    print("trace(log(Q)) ≈", est)
    print("info:", info)
    print(f"Elapsed: {t1 - t0:.3f} s")
    print("Max adaptive Leja degree used:", info["leja_m_max"])

Loaded Q: (999999, 999999) nnz = 4995991
trace(log(Q)) ≈ 3394586.0375791974
info: {'m_S': 4, 'm_Q': 4, 'm_G': 4, 'trace_proj': 124.3164508238623, 'trace_resid': 31025455.206035804, 'lambda_min': 1e-12, 'lambda_max': 80.0, 'alpha_scale': 1e-12, 'a_scaled': 1.0, 'b_scaled': 80000000000000.0, 'shift_nlogalpha': -27630993.48490743, 'used_matvec_budget': 12, 'leja_m_used': [87, 87, 87, 87, 64, 64, 64, 64, 87, 87, 87, 87], 'leja_m_max': 87}
Elapsed: 9.073 s
Max adaptive Leja degree used: 87
